<a href="https://colab.research.google.com/github/DhairyaDave08/BetaData-SpaceX/blob/main/Notebooks/02_Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **02 — Feature Engineering**

Goal: build leak-free predictive features for the mission success model.

Core principle: every feature for a launch at time T must only use information
available strictly BEFORE T. This directly satisfies the problem statement's
"no future leakage" constraint and is what makes our calibration trustworthy.

This notebook:
1. Reloads and cleans the raw dataset (self-contained — no dependency on prior notebook's session state)
2. Adds a payload capacity feature (via public-knowledge lookup, since the dataset has no payload mass column)
3. Builds leak-free rolling reliability features per rocket family, launch site, and country
4. Groups rare/high-cardinality categories to prevent overfitting on this small (~4,200 row) dataset
5. Saves the final feature set to `data/processed/features.csv`

In [8]:
!git clone https://github.com/DhairyaDave08/BetaData-SpaceX.git
%cd BetaData-SpaceX
!pip install -q -r requirements.txt

import sys
sys.path.append('/content/BetaData-SpaceX')

import pandas as pd
import numpy as np
import os

pd.set_option("display.max_columns", None)

Cloning into 'BetaData-SpaceX'...
remote: Enumerating objects: 48, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 48 (delta 14), reused 16 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (48/48), 309.24 KiB | 1.76 MiB/s, done.
Resolving deltas: 100% (14/14), done.
/content/BetaData-SpaceX/BetaData-SpaceX


## 1. Reload and clean raw data

Colab sessions don't persist files between notebooks, so we regenerate
`launches_clean.csv` here rather than assuming it already exists.

In [9]:
from src.data_loader import load_raw_data, clean_dataset

os.makedirs("data/processed", exist_ok=True)

raw = load_raw_data("Data/Raw/Space_Corrected.csv")  # match your actual repo path/casing
df = clean_dataset(raw)
df.to_csv("data/processed/launches_clean.csv", index=False)

print(f"Cleaned dataset shape: {df.shape}")
df.head(3)

Dropped 126 rows with missing target/date out of 4324
Cleaned dataset shape: (4198, 16)


,company_name,location,datum,detail,status_rocket,rocket,status_mission,launch_date,year,decade,rocket_family,payload_desc,country,launch_site,cost_musd,mission_success
0,RVSN USSR,"Site 1/5, Baikonur Cosmodrome, Kazakhstan","Fri Oct 04, 1957 19:28 UTC",Sputnik 8K71PS | Sputnik-1,StatusRetired,NaN,Success,1957-10-04 19:28:00,1957.0,1950.0,Sputnik 8K71PS,Sputnik-1,Kazakhstan,Site 1/5,NaN,1
1,RVSN USSR,"Site 1/5, Baikonur Cosmodrome, Kazakhstan","Sun Nov 03, 1957 02:30 UTC",Sputnik 8K71PS | Sputnik-2,StatusRetired,NaN,Success,1957-11-03 02:30:00,1957.0,1950.0,Sputnik 8K71PS,Sputnik-2,Kazakhstan,Site 1/5,NaN,1
2,US Navy,"LC-18A, Cape Canaveral AFS, Florida, USA","Fri Dec 06, 1957 16:44 UTC",Vanguard | Vanguard TV3,StatusRetired,NaN,Failure,1957-12-06 16:44:00,1957.0,1950.0,Vanguard,Vanguard TV3,USA,LC-18A,NaN,0


## 2.Sort chronologically

This is non-negotiable. Every rolling/leak-free feature downstream assumes
the data is in strict launch-date order. We assert this before proceeding.

In [10]:
df = df.sort_values("launch_date").reset_index(drop=True)
assert df["launch_date"].is_monotonic_increasing, "Data must be sorted by launch_date!"
print("Sorted and verified chronological order.")
print(f"Date range: {df['launch_date'].min().date()} to {df['launch_date'].max().date()}")

Sorted and verified chronological order.
Date range: 1957-10-04 to 2020-08-07


## 3. Payload capacity feature

The dataset has no payload mass column. Rather than skip this objective
entirely, we use a static lookup of publicly known approximate LEO payload
capacities per rocket family. This is documented explicitly as an assumption
in `data/DATA_SOURCES.md` — it is not fabricated per-launch data, just a
reasonable proxy for "how large a rocket this is."

Rocket families not found in the lookup get a fallback value (the median
of known capacities) and are flagged via `payload_capacity_known = False`,
so the model and any downstream analysis can distinguish real values from
fallback values.

In [11]:
from src.feature_engineering import add_payload_capacity

df = add_payload_capacity(df)

print(f"Rows with known payload capacity: {df['payload_capacity_known'].sum()} / {len(df)} "
      f"({df['payload_capacity_known'].mean():.1%})")

df[["rocket_family", "payload_capacity_kg", "payload_capacity_known"]].sample(10, random_state=42)

Rows with known payload capacity: 3165 / 4198 (75.4%)


,rocket_family,payload_capacity_kg,payload_capacity_known
2691,Delta II 7925-10,6100,True
2034,Cosmos-3M (11K65M),1400,True
3552,Atlas V 541,18850,True
3744,H-IIA 202,10000,True
298,Voskhod,5700,True
2618,Molniya-M /Block ML,1600,True
2530,Delta II 7925,6100,True
1509,Tsyklon-2,4000,True
1207,Soyuz M,7020,True
2408,Molniya-M /Block 2BL,1600,True


## 4. Leak-free rolling reliability features

For each launch, we compute:
- **prior_flights**: how many times this rocket family / site / country has flown *before* this launch
- **prior_success_rate**: the success rate of all *prior* flights only (current and future outcomes excluded)

This is implemented using `.shift(1)` before `.expanding().mean()` — shifting
by 1 row ensures a launch's own outcome never contributes to its own feature.
Rows with no prior history (a rocket family's very first flight) fall back to
the global historical success rate rather than 0 or 1, since there's no
information yet to justify an extreme value.

We validate this logic with a unit test in `tests/test_no_leakage.py`.

In [12]:
from src.feature_engineering import add_rolling_reliability_features

df = add_rolling_reliability_features(df)

df[[
    "launch_date", "rocket_family", "mission_success",
    "vehicle_prior_flights", "vehicle_prior_success_rate"
]].head(15)

,launch_date,rocket_family,mission_success,vehicle_prior_flights,vehicle_prior_success_rate
0,1957-10-04 19:28:00,Sputnik 8K71PS,1,0,0.903764
1,1957-11-03 02:30:00,Sputnik 8K71PS,1,1,1.000000
2,1957-12-06 16:44:00,Vanguard,0,0,0.903764
3,1958-02-01 03:48:00,Juno I,1,0,0.903764
4,1958-02-05 07:33:00,Vanguard,0,1,0.000000
5,1958-03-05 18:27:00,Juno I,0,1,1.000000
6,1958-03-17 12:15:00,Vanguard,1,2,0.000000
7,1958-03-26 17:38:00,Juno I,1,2,0.500000
8,1958-04-27 09:01:00,Sputnik 8A91,0,0,0.903764
9,1958-04-28 02:53:00,Vanguard,0,3,0.333333


### Sanity check on rolling features

Let's manually verify this is behaving correctly for a well-known, frequently
launched rocket — Falcon 9. Prior success rate should generally trend upward
over time as SpaceX's reliability matured, and no row should reflect an
outcome from a launch that hasn't happened yet.

In [13]:
falcon9 = df[df["rocket_family"].str.contains("Falcon 9", case=False, na=False)]
falcon9[[
    "launch_date", "mission_success",
    "vehicle_prior_flights", "vehicle_prior_success_rate"
]].head(20)

,launch_date,mission_success,vehicle_prior_flights,vehicle_prior_success_rate
3494,2010-06-04 18:45:00,1,0,0.903764
3511,2010-12-08 15:43:00,1,1,1.000000
3565,2012-05-22 07:44:00,1,2,1.000000
3581,2012-10-08 00:35:00,0,3,1.000000
3597,2013-03-01 15:10:00,1,4,0.750000
3624,2013-09-29 16:00:00,1,0,0.903764
3630,2013-12-03 22:41:00,1,1,1.000000
3636,2014-01-06 22:06:00,1,2,1.000000
3647,2014-04-18 19:25:00,1,3,1.000000
3661,2014-07-14 15:15:00,1,4,1.000000


### Run the automated leak-check test

This is a formal unit test (not just eyeballing), confirming the shift-based
logic is mathematically leak-free on a small hand-verified example.

In [14]:
!python tests/test_no_leakage.py

PASSED: no-leakage rolling feature test


## 5. Vehicle maturity feature

Young rockets (early in their flight history) are known to fail more often
than mature, well-tested vehicles. We capture this with `vehicle_age_days`:
days elapsed since this rocket family's very first recorded launch.

In [15]:
first_launch = df.groupby("rocket_family")["launch_date"].transform("min")
df["vehicle_age_days"] = (df["launch_date"] - first_launch).dt.days

df[["rocket_family", "launch_date", "vehicle_age_days"]].sort_values("vehicle_age_days").head(5)

,rocket_family,launch_date,vehicle_age_days
1085,Molniya-M /Block 2BL,1972-09-19 19:19:00,0
2739,Athena I,1995-08-15 22:30:00,0
4170,LauncherOne,2020-05-25 19:50:00,0
538,Molniya-M /Block VL,1967-06-12 02:39:00,0
3775,Long March 7/YZ-1A,2016-06-25 12:00:00,0


## 6. Handle high-cardinality categoricals

We found **339 unique rocket families**, but only 20 of them account for
48.5% of all launches — a long tail of 199 families with 5 or fewer launches
each. One-hot encoding all 339 would badly overfit a ~4,200-row dataset.

We tested several thresholds for grouping rare categories into `"other"`:

| min_count | families kept | coverage |
|-----------|---------------|----------|
| 10        | 91            | 80.5%    |
| 15        | 55            | 69.6%    |
| 20        | 47            | 66.3%    |
| 25        | 36            | 60.3%    |

**`min_count=10`** is the best tradeoff — it keeps meaningfully more coverage
(80.5%) without an excessive category count, and XGBoost (tree-based) handles
~91 one-hot categories on this dataset size without difficulty.

We apply the same `min_count=10` threshold to launch site and country.

In [16]:
from src.feature_engineering import group_rare_categories

df = group_rare_categories(df, "rocket_family", min_count=10)
df = group_rare_categories(df, "launch_site", min_count=10)
df = group_rare_categories(df, "country", min_count=10)

print("Rocket families:", df["rocket_family"].nunique(), "->", df["rocket_family_grouped"].nunique())
print("Launch sites:   ", df["launch_site"].nunique(), "->", df["launch_site_grouped"].nunique())
print("Countries:      ", df["country"].nunique(), "->", df["country_grouped"].nunique())

Rocket families: 339 -> 98
Launch sites:    126 -> 69
Countries:       21 -> 10


## 7. Assemble final feature set

We select only the columns the model will actually use, drop intermediate
helper columns, and save to `data/processed/features.csv` — the input for
the training notebook.

In [17]:
feature_cols = [
    "launch_date", "year", "decade", "mission_success",
    "rocket_family_grouped", "launch_site_grouped", "country_grouped",
    "payload_capacity_kg", "payload_capacity_known",
    "vehicle_prior_flights", "vehicle_prior_success_rate",
    "site_prior_flights", "site_prior_success_rate",
    "country_prior_flights", "country_prior_success_rate",
    "vehicle_age_days",
]

features = df[feature_cols].copy()
features.to_csv("data/processed/features.csv", index=False)

print(f"Final feature set shape: {features.shape}")
features.head(10)

Final feature set shape: (4198, 16)


,launch_date,year,decade,mission_success,rocket_family_grouped,launch_site_grouped,country_grouped,payload_capacity_kg,payload_capacity_known,vehicle_prior_flights,vehicle_prior_success_rate,site_prior_flights,site_prior_success_rate,country_prior_flights,country_prior_success_rate,vehicle_age_days
0,1957-10-04 19:28:00,1957.0,1950.0,1,other,Site 1/5,Kazakhstan,10750,False,0,0.903764,0,0.903764,0,0.903764,0
1,1957-11-03 02:30:00,1957.0,1950.0,1,other,Site 1/5,Kazakhstan,10750,False,1,1.000000,1,1.000000,1,1.000000,29
2,1957-12-06 16:44:00,1957.0,1950.0,0,Vanguard,LC-18A,USA,10750,False,0,0.903764,0,0.903764,0,0.903764,0
3,1958-02-01 03:48:00,1958.0,1950.0,1,other,other,USA,10750,False,0,0.903764,0,0.903764,1,0.000000,0
4,1958-02-05 07:33:00,1958.0,1950.0,0,Vanguard,LC-18A,USA,10750,False,1,0.000000,1,0.000000,2,0.500000,60
5,1958-03-05 18:27:00,1958.0,1950.0,0,other,other,USA,10750,False,1,1.000000,1,1.000000,3,0.333333,32
6,1958-03-17 12:15:00,1958.0,1950.0,1,Vanguard,LC-18A,USA,10750,False,2,0.000000,2,0.000000,4,0.250000,100
7,1958-03-26 17:38:00,1958.0,1950.0,1,other,LC-5,USA,10750,False,2,0.500000,0,0.903764,5,0.400000,53
8,1958-04-27 09:01:00,1958.0,1950.0,0,other,Site 1/5,Kazakhstan,10750,False,0,0.903764,2,1.000000,2,1.000000,0
9,1958-04-28 02:53:00,1958.0,1950.0,0,Vanguard,LC-18A,USA,10750,False,3,0.333333,3,0.333333,6,0.500000,142


## 8. Final validation before moving to modeling

In [18]:
print("Missing values per column:")
print(features.isna().sum())

print(f"\nGrouped rocket families: {features['rocket_family_grouped'].nunique()}")
print(f"Grouped launch sites:    {features['launch_site_grouped'].nunique()}")
print(f"Grouped countries:       {features['country_grouped'].nunique()}")
print(f"\nTarget balance — success rate: {features['mission_success'].mean():.2%}")
print(f"Date range: {features['launch_date'].min()} to {features['launch_date'].max()}")

Missing values per column:
launch_date                   0
year                          0
decade                        0
mission_success               0
rocket_family_grouped         0
launch_site_grouped           0
country_grouped               0
payload_capacity_kg           0
payload_capacity_known        0
vehicle_prior_flights         0
vehicle_prior_success_rate    0
site_prior_flights            0
site_prior_success_rate       0
country_prior_flights         0
country_prior_success_rate    0
vehicle_age_days              0
dtype: int64

Grouped rocket families: 98
Grouped launch sites:    69
Grouped countries:       10

Target balance — success rate: 90.38%
Date range: 1957-10-04 19:28:00 to 2020-08-07 05:12:00


## Summary

- Built leak-free rolling reliability features for rocket family, launch site,
  and country, verified by unit test.
- Added a documented payload-capacity proxy feature to satisfy the payload-mass
  objective despite the dataset lacking that column natively.
- Reduced 339 rocket families to ~91 groups (min_count=10), preserving 80.5%
  of raw category-level signal while avoiding overfitting on a small dataset.
- Output saved to `data/processed/features.csv`, ready for era-stratified
  train/test split and calibrated model training in the next notebook.

**Next:** `03_modeling_baseline.ipynb` — Logistic Regression baseline with
era-stratified split.